# Dyehouse — Data Cleaning & Normalization

Analyze and normalize the `database-dyehouse-sic-jac.csv'` reference data from SIC JAC.
Focus on columns A (codigo), B (descripcion), and L (SUB-CATEGORIA).

**Source:** `docs/research/data/data/database-dyehouse-sic-jac.csv'`

In [4]:
# Leer el archivo completo como texto
import os
print(f"Working directory: {os.getcwd()}")

# Usar ruta absoluta
abs_path = r"c:\Users\Alexander\Desktop\illampu\yarn-epr\docs\research\data\database-dyehouse-sic-jac.csv"
with open(abs_path, 'r', encoding='utf-8') as f:
    contenido = f.read()
print(contenido)

Working directory: c:\Users\Alexander\Desktop\illampu\yarn-epr\docs\analysis\warehouse
CATEGORIAS
1. Colorantes
2. Insumos quimicos
SUB CATEGORIAS DE COLORANTES:
1. COLORANTES 
2. COLORANTES ANT 
SUB CATEGORIAS DE INSUMOS QUIMICOS:
1. QUIMICOS
ITEMS:
4325C12   PARAFINA 4325C12
4325R14   PARAFINA 4325R14
COL-01    AZUL MARINO HISPACRYL CMT 265% COLORANTES
COL-02    ROJO HISPACRYL GTL 200% COLORANTES
COL-03    AMARILLO FLAVINA HISPACRYL 10GF 300% COLORANTES
COL-04    VERDE VICTORIA WB 170% COLORANTES
COL-05    AMARILLO ORO XGL 250% COLORANTES
COL-06    CATENIOC VIOLET X 52% COLORANTES
COL-07    AMARILLO ERICROMO COLORANTES
COL-08    ROJO ERICROMO 180% COLORANTES
COL-09    BLUE BASIC X-GRL COLORANTES
COL-10    TURQUEZA XBG 250%
COL-11    VIOLET BASIC 1G 330% COLORANTES
COL-12    ROJO BASIC GRL 250% COLORANTES
COL-13    ROJO BTE HISPACRIL 4G 250% COLORANTES
COL-14    ROJO HISPACRIL GRL 270% COLORANTES
COL-15    AMARILLO ORO HISPACRIL 200% COLORANTES
COL-16    ROJO BTE HISPACRIL 2B 165% COL

In [5]:
# Extraer los items y separarlos por prefijo de código
items_section = contenido.split("ITEMS:")[1]

# Diccionario para agrupar items por prefijo
items_by_prefix = {}

# Procesar cada línea
for line in items_section.strip().split('\n'):
    line = line.strip()
    if line:
        # Extraer el prefijo (BLS-, ETQ-, DEV-, TAL-, etc.)
        parts = line.split()
        if parts:
            codigo = parts[0]
            prefijo = codigo.split('-')[0]
            
            if prefijo not in items_by_prefix:
                items_by_prefix[prefijo] = []
            
            items_by_prefix[prefijo].append(line)

# Mostrar items organizados por prefijo
for prefijo in sorted(items_by_prefix.keys()):
    print(f"\n{'='*60}")
    print(f"PREFIJO: {prefijo}")
    print(f"{'='*60}")
    for item in items_by_prefix[prefijo]:
        print(item)


PREFIJO: 4325C12
4325C12   PARAFINA 4325C12

PREFIJO: 4325R14
4325R14   PARAFINA 4325R14

PREFIJO: COL
COL-01    AZUL MARINO HISPACRYL CMT 265% COLORANTES
COL-02    ROJO HISPACRYL GTL 200% COLORANTES
COL-03    AMARILLO FLAVINA HISPACRYL 10GF 300% COLORANTES
COL-04    VERDE VICTORIA WB 170% COLORANTES
COL-05    AMARILLO ORO XGL 250% COLORANTES
COL-06    CATENIOC VIOLET X 52% COLORANTES
COL-07    AMARILLO ERICROMO COLORANTES
COL-08    ROJO ERICROMO 180% COLORANTES
COL-09    BLUE BASIC X-GRL COLORANTES
COL-10    TURQUEZA XBG 250%
COL-11    VIOLET BASIC 1G 330% COLORANTES
COL-12    ROJO BASIC GRL 250% COLORANTES
COL-13    ROJO BTE HISPACRIL 4G 250% COLORANTES
COL-14    ROJO HISPACRIL GRL 270% COLORANTES
COL-15    AMARILLO ORO HISPACRIL 200% COLORANTES
COL-16    ROJO BTE HISPACRIL 2B 165% COLORANTES
COL-17    AMARILLO HISPACRIL 8GL 245% COLORANTES
COL-18    AZUL HISPACRIL GRL 300% COLORANTES
COL-19    VERDE MALAQUITA COLORANTES
COL-20    AZUL HISPACRIL 5G 300% COLORANTES
COL-21    YELLOW B

## 1. Parse y filtrar colores, concentraciones y códigos

Extraer a partir de `descripcion` los campos clave para análisis: `color`, `concentracion`, `codigo_prefijo` y `es_recuperado`.

In [6]:
import re
import pandas as pd
from pathlib import Path

# Convertir lista de items a DataFrame estructurado
rows = []
for prefijo, items in items_by_prefix.items():
    for item in items:
        parts = item.split(None, 1)
        codigo = parts[0]
        descripcion = parts[1] if len(parts) > 1 else ''
        rows.append({'codigo': codigo, 'descripcion': descripcion})

df = pd.DataFrame(rows)

# Parse helpers

def parse_color(desc):
    if not isinstance(desc, str) or not desc.strip():
        return None
    partes = re.split(r'\s{2,}', desc.strip())
    return partes[0].strip() if partes else desc.strip()


def parse_codigo_prefijo(codigo):
    if not isinstance(codigo, str):
        return None
    return codigo.split('-', 1)[0]


def parse_concentracion(desc):
    if not isinstance(desc, str):
        return None
    match = re.search(r'(\d+/\d+)', desc)
    return match.group(1) if match else None


# Agregar columnas de análisis
[df['color'], df['codigo_prefijo'], df['concentracion'], df['es_recuperado']] = (
    df['descripcion'].apply(parse_color),
    df['codigo'].apply(parse_codigo_prefijo),
    df['descripcion'].apply(parse_concentracion),
    df['descripcion'].str.contains(r'RECUPER|MATERI', case=False, na=False, regex=True)
)

# Mostrar resultados y ejemplos
print('Columnas creadas: ', df.columns.tolist())
print('\nEjemplo de filas parseadas:')
print(df[['codigo', 'descripcion', 'color', 'concentracion', 'codigo_prefijo', 'es_recuperado']].head(20).to_string(index=False))

print('\nPrefijos de código:')
print(df['codigo_prefijo'].value_counts())

print('\nConcentraciones detectadas:')
print(df['concentracion'].value_counts())

print('\nColores más frecuentes:')
print(df['color'].value_counts().head(25))

Columnas creadas:  ['codigo', 'descripcion', 'color', 'codigo_prefijo', 'concentracion', 'es_recuperado']

Ejemplo de filas parseadas:
 codigo                                     descripcion                                           color concentracion codigo_prefijo  es_recuperado
4325C12                                PARAFINA 4325C12                                PARAFINA 4325C12          None        4325C12          False
4325R14                                PARAFINA 4325R14                                PARAFINA 4325R14          None        4325R14          False
 COL-01       AZUL MARINO HISPACRYL CMT 265% COLORANTES       AZUL MARINO HISPACRYL CMT 265% COLORANTES          None            COL          False
 COL-02              ROJO HISPACRYL GTL 200% COLORANTES              ROJO HISPACRYL GTL 200% COLORANTES          None            COL          False
 COL-03 AMARILLO FLAVINA HISPACRYL 10GF 300% COLORANTES AMARILLO FLAVINA HISPACRYL 10GF 300% COLORANTES          None        

In [7]:
# Filtros útiles

df_azul = df[df['color'].str.contains('AZUL', case=False, na=False)]
df_2_18 = df[df['concentracion'] == '2/18']
df_rec = df[df['codigo_prefijo'] == 'REC']
df_mat = df[df['codigo_prefijo'] == 'MAT']

df_recuperados = df[df['es_recuperado']]

print('Filtrado AZUL:', len(df_azul))
print('Filtrado 2/18:', len(df_2_18))
print('Filtrado REC:', len(df_rec))
print('Filtrado MAT:', len(df_mat))
print('Recuperados:', len(df_recuperados))

# Agrupar por color, concentración y prefijo
agrupado = (
    df.groupby(['color', 'concentracion', 'codigo_prefijo'])
    .size()
    .reset_index(name='count')
    .sort_values(['count', 'color'], ascending=[False, True])
)
print('\nPrimeras filas del agrupado:')
print(agrupado.head(30).to_string(index=False))

Filtrado AZUL: 4
Filtrado 2/18: 0
Filtrado REC: 0
Filtrado MAT: 0
Recuperados: 0

Primeras filas del agrupado:
Empty DataFrame
Columns: [color, concentracion, codigo_prefijo, count]
Index: []
